# Partie 1 — SQL (DuckDB) sur PaySim
Une ligne = une transaction ; `step` = 1 heure (743 steps ≈ 31 jours).

In [ ]:
from pathlib import Path
import duckdb

con = duckdb.connect()
con.sql("SET enable_progress_bar = false")
con.sql("""
CREATE OR REPLACE TABLE tx AS
SELECT row_number() OVER () AS tx_id, *
FROM read_csv('../../PS_20174392719_1491204439457_log.csv')
""")
con.sql("SELECT count(*) AS n, sum(isFraud) AS fraudes, round(100 * avg(isFraud), 3) AS pct_fraude FROM tx").df()

## 1. Contrôle de qualité : doublons et volumétrie par jour

In [ ]:
con.sql("""
SELECT count(*) - count(DISTINCT (step, type, amount, nameOrig, nameDest)) AS doublons FROM tx
""").df()

In [ ]:
con.sql("""
WITH par_jour AS (
    SELECT (step - 1) // 24 + 1 AS jour, count(*) AS n, sum(isFraud) AS fraudes
    FROM tx GROUP BY jour
)
SELECT jour, n, fraudes, round(100.0 * fraudes / n, 2) AS pct_fraude,
       round(100.0 * (n - lag(n) OVER (ORDER BY jour)) / lag(n) OVER (ORDER BY jour), 1) AS evol_n_pct
FROM par_jour ORDER BY jour
""").df()

Aucun doublon. En revanche, le volume s'effondre les jours 3-5 et à partir du jour 18 alors que les fraudes restent stables (~270/jour) :
artefact de simulation → `step` ne sera pas une variable explicative.

## 2. Agrégation : fraude par type

In [ ]:
con.sql("""
SELECT type, count(*) AS n, sum(isFraud) AS fraudes, round(100 * avg(isFraud), 3) AS pct_fraude,
       sum(isFlaggedFraud) AS alertes_regle
FROM tx GROUP BY type ORDER BY fraudes DESC
""").df()

Fraude uniquement sur `TRANSFER` et `CASH_OUT` → périmètre de modélisation. La règle actuelle ne lève que 16 alertes.

## 3. Variable de fenêtre sans fuite
Nombre de transactions du compte émetteur et du compte destinataire sur les steps strictement précédents (`1 PRECEDING`).
Calculé sur tout le fichier, avant échantillonnage.

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE fenetre AS
SELECT tx_id,
       count(*) OVER (PARTITION BY nameOrig ORDER BY step
                      RANGE BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS nb_tx_orig_avant,
       count(*) OVER (PARTITION BY nameDest ORDER BY step
                      RANGE BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS nb_recu_dest_avant
FROM tx
""")
con.sql("""
SELECT isFraud, round(avg(nb_tx_orig_avant), 4) AS moy_orig_avant,
       round(100 * avg((nb_recu_dest_avant = 0)::INT), 1) AS pct_dest_nouveau
FROM tx JOIN fenetre USING (tx_id)
WHERE type IN ('TRANSFER', 'CASH_OUT') GROUP BY isFraud
""").df()

Les émetteurs sont quasi uniques (historique ≈ 0) ; côté destinataire, 63 % des fraudes vont vers un compte nouveau contre 15 % sinon.

## 4. Échantillon pour la suite (sans colonnes fuyantes `newbalance*`, `isFlaggedFraud`)

In [ ]:
con.sql("SET threads = 1")  
con.sql("""
COPY (
    SELECT * FROM (
        SELECT tx_id, step, (step - 1) % 24 AS heure, type, amount, oldbalanceOrg, oldbalanceDest,
               nb_tx_orig_avant, nb_recu_dest_avant, isFraud
        FROM tx JOIN fenetre USING (tx_id)
        WHERE type IN ('TRANSFER', 'CASH_OUT') ORDER BY tx_id
    ) USING SAMPLE 20 PERCENT (bernoulli, 42)
) TO 'paysim_echantillon.parquet' (FORMAT parquet)
""")
con.sql("SELECT count(*) AS n, sum(isFraud) AS fraudes FROM 'paysim_echantillon.parquet'").df()